# Tui Voice Mini Malayalam
Run top to bottom on an A100/L4/T4 runtime. Stop at any failed gate and paste the output back.

In [ ]:
GITHUB_USER='Cyrax321'
!nvidia-smi
from google.colab import drive; drive.mount('/content/drive')
import os
WORK='/content/drive/MyDrive/tui_voice'; os.makedirs(WORK,exist_ok=True)   # Drive: reports and checkpoints only
DATA='/content/data'; os.makedirs(DATA,exist_ok=True)                            # local SSD: wavs, features
!apt-get -qq install -y espeak-ng
!git clone https://github.com/n4ze3m/tiny-voca.git /content/tiny-voca
!git clone https://github.com/{GITHUB_USER}/tui-voice.git /content/tui-voice
!pip -q install -e "/content/tiny-voca[own]" -U huggingface_hub -r /content/tui-voice/requirements.txt
%cd /content/tiny-voca
!hf download n4ze3m/tiny-voca --local-dir weights || huggingface-cli download n4ze3m/tiny-voca --local-dir weights
os.environ['DIZIONE_DATA']='/content/tiny-voca/weights/phonemizer'

## Gate 0: inspect upstream (paste `upstream_report.txt` back)

In [ ]:
!python /content/tui-voice/scripts/00_inspect_upstream.py {WORK}/upstream_report.txt
!cp upstream_symbols.json {WORK}/ 2>/dev/null; ls {WORK}
!head -c 6000 {WORK}/upstream_report.txt

## Data: accept terms at huggingface.co/datasets/ai4bharat/Rasa, then login

In [ ]:
import os, getpass
from huggingface_hub import login
if 'HF_TOKEN' not in os.environ:
    tok = getpass.getpass('Paste your Hugging Face Token: ').strip()
    login(token=tok)
    os.environ['HF_TOKEN'] = tok
%cd /content/tui-voice
!git pull origin main
!python scripts/01_prepare_rasa.py --list

In [ ]:
CFG='Malayalam'
!python scripts/01_prepare_rasa.py --config {CFG} --out {DATA}

## Gate 1: phonemes. `upstream_symbols.json` is written by Gate 0 from weights/config.json; check it has about 76 symbols

In [ ]:
!python scripts/02_phonemize.py --data {DATA} --upstream-symbols {WORK}/upstream_symbols.json

## Gate 2: alignment (try 200 clips first, listen to 10)

In [ ]:
!python scripts/03_align.py --data {DATA} --out {DATA}/annotations --limit 200
!python scripts/04_check_annotations.py {DATA}/annotations

In [ ]:
# full run once the 200-clip check looks right
!python scripts/03_align.py --data {DATA} --out {DATA}/annotations
!python scripts/04_check_annotations.py {DATA}/annotations

Alignments only describe durations. tiny-voca trains on precomputed feature files (mel, prosody, wave) made by `tiny_voca.prepare`, and expects speaker names from its config (F2/M2), so `03_align.py` writes speaker `F2` by default and the bridge must run `prepare` on our 24 kHz wavs and point `--acoustic-root` at the result.

## Train (bridge pending Gate 0)
Written once `upstream_report.txt` shows how tiny-voca loads checkpoints, builds features and checks the annotation fingerprint.

In [ ]:
# Step 1: Extract features, remap checkpoint, patch annotations, and launch training
# This wraps tiny_voca.prepare + init_ckpt remap + tiny_voca.train in one script
%cd /content/tiny-voca
!python /content/tui-voice/scripts/06_train_bridge.py \
  --annotations {DATA}/annotations \
  --wavs {DATA}/wavs \
  --upstream-ckpt weights/tiny-voca.pt \
  --vocab-plan {DATA}/vocab_plan.json \
  --config weights/config.json \
  --output {WORK}/run1 \
  --steps 40000 --gan-start 8000 \
  --save-every 1000 --keep-every 5000 \
  --batch-frames 9000 --learning-rate 3e-4

## Export & Publish to Hugging Face
Packages the best checkpoint, config with Malayalam symbols, model card, and ONNX export, then pushes to Hugging Face Hub.

In [ ]:
%cd /content/tui-voice
HF_REPO = f"{GITHUB_USER}/tui-voice-mini-ml"
!python scripts/05_push_to_hf.py --repo-id {HF_REPO} --checkpoint {WORK}/run1/last.pt --config /content/tiny-voca/weights/config.json --vocab-plan {DATA}/vocab_plan.json --out {WORK}/export
